<a href="https://colab.research.google.com/github/turayusa/homework/blob/main/homework_(3)_ipynb_Fundermantal_of_Maths.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DX 601 Week 3 Homework

## Introduction

In this homework, you will practice plotting data and calculating model predictions and losses.

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for sample code.

* https://github.com/bu-cds-omds/dx500-examples
* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Instructions

You should replace every instance of "..." or "TODO" below.
These are where you are expected to write code to answer each problem.

After some of the problems, there are extra code cells that will test functions that you wrote so you can quickly see how they run on an example.
If your code works on these examples, it is more likely to be correct.
However, the autograder will test different examples, so working correctly on these examples does not guarantee full credit for the problem.
You may change the example inputs to further test your functions on your own.
You may also add your own example inputs for problems where we did not provide any.

Be sure to run each code block after you edit it to make sure it runs as expected.
When you are done, we strongly recommend you run all the code from scratch (Runtime menu -> Restart and Run all) to make sure your current code works for all problems.

If your code raises an exception when run from scratch, it will  interfere with the auto-grader process causing you to lose some or all points for this homework.
Please ask for help in YellowDig or schedule an appointment with a learning facilitator if you get stuck.


## Shared Imports

Do not install or use any additional modules.
Installing additional modules may result in an autograder failure resulting in zero points for some or all problems.

In [64]:
from collections import defaultdict
from enum import Enum
from typing import Dict, List, Tuple, Type
from pprint import pprint
import csv
import os
import numpy as np

# The Data

In this assignment you will be training a probabilistic model to predict whether an avocado is good to eat (or not) given a feature description of the avocado.


### Problem 1: Storing Conditional Probability Mass Functions

The code below creates some enums for the features we will be using in this homework. Please do not change this


In [65]:
class Color(Enum):
    BLACK = 0
    BROWN = 1
    GREEN = 2

    @classmethod
    def value_of(cls,
                 s: str
                 ) -> "Color":
        for x in cls._member_names_:
            if s.upper() == x:
                return cls.__dict__[x]
        raise ValueError(f"ERROR: unknown string {s}")



class Softness(Enum):
    MUSHY = 0
    SOFT = 1
    TENDER = 2
    HARD = 3

    @classmethod
    def value_of(cls,
                 s: str
                 ) -> "Softness":
        for x in cls._member_names_:
            if s.upper() == x:
                return cls.__dict__[x]
        raise ValueError(f"ERROR: unknown string {s}")


class GoodToEat(Enum):
    YES = 0
    NO = 1

    @classmethod
    def value_of(cls,
                 s: str
                 ) -> "GoodToEat":
        for x in cls._member_names_:
            if s.upper() == x:
                return cls.__dict__[x]
        raise ValueError(f"ERROR: unknown string {s}")

Below is some code to load the accopanying data file. Please do not change this

In [66]:
def load_data() -> List[Tuple[Color, Softness, GoodToEat]]:
    data_file: str = os.path.join("train_avacados.txt")
    if not os.path.exists(data_file):
        raise Exception(f"ERROR: file {data_file} does not exist!")

    data: List[Tuple[Color, Softness, GoodToEat]] = list()
    with open(data_file, "r") as f:
        reader = csv.reader(f, delimiter=",")

        for line in reader:
            if len(line) > 0:

                if len(line) != 3:
                    raise ValueError(f"ERROR: expected three values but got {line}")

                color, softness, good_to_eat = line
                data.append(tuple([Color.value_of(color.strip().rstrip()),
                                   Softness.value_of(softness.strip().rstrip()),
                                   GoodToEat.value_of(good_to_eat.strip().rstrip())]))

    return data

In [67]:
data = load_data()
pprint(data)

[(<Color.BLACK: 0>, <Softness.MUSHY: 0>, <GoodToEat.NO: 1>),
 (<Color.BLACK: 0>, <Softness.MUSHY: 0>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <Softness.MUSHY: 0>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.BROWN: 1>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.BROWN: 1>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.BROWN: 1>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.TENDER: 2>, <GoodToEat.YES: 0>),
 (<Color.BLACK: 0>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.HARD: 3>, <GoodToEat.NO: 1>),
 (<Color.GREEN: 2>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.HARD: 3>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <Softness.HARD: 3>, <GoodToEat.NO: 1>),
 (<Color.GREEN: 2>, <Softness.TENDER: 2>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <Softness.TENDER: 2>, <GoodToEat.YES: 0>),
 (<Color.BLACK: 0>, <Softness.SOFT: 1>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <S

Below is the beginning of a class. Complete the `fit` method to populate the three fields in this class from the training data. The field `good_to_eat_prior` should contain a probability mass function over the `GoodToEat` feature. You should calculate it as follows:

$$Pr[GoodToEat=a] = \dfrac{\#\text{ of times}(GoodToEat=a)}{\sum\limits_{b\in GoodToEat} \#\text{ of times}(GoodToEat=b)} $$

This field is a dictionary where the key is the value of `GoodToEat` and the value of the dictionary is the probability.

The field `softness_given_good_to_eat_pmf` should contain the following conditional probability distribution:
$Pr[Softness | GoodToEat]$. This distribution is stored as a dictionary of dictionaries. A key to the outer dictionary is a value of `GoodToEat` and using this key returns a pmf over the `Softness` feature. Therefore if you wanted to know a particular value
$Pr[Softness=a | GoodToEat=b]$ you would look it up in this field with the code `self.softness_given_good_to_eat_pmf[b][a]`. You should calculate these pmfs as follows:

$$Pr[Softness = a | GoodToEat = b]= \dfrac{\#\text{ of times}(Softness = a \cap GoodToEat=b)}{\#\text{ of times}(GoodToEat=b)}$$

Likewise you should follow a similar procedure to populate the `color_give_good_to_eat_pmf` field

In [68]:
# YOUR CHANGES HERE

class AvocadoPredictor(object):
    def __init__(self) -> None:
        self.color_given_good_to_eat_pmf: Dict[GoodToEat, Dict[Color, float]] = defaultdict(lambda: defaultdict(float))
        self.softness_given_good_to_eat_pmf: Dict[GoodToEat, Dict[Softness, float]] = defaultdict(lambda: defaultdict(float))
        self.good_to_eat_prior: Dict[GoodToEat, float] = defaultdict(float)

    """
    Problem 1: populate the fields from the constructor with the provided training data
    """
    def fit(self,
            data: List[Tuple[Color, Softness, GoodToEat]]
            ) -> "AvocadoPredictor":

        good_counts = defaultdict(int)
        color_counts = defaultdict(lambda: defaultdict(int))
        softness_counts = defaultdict(lambda: defaultdict(int))

        for color, softness, good_to_eat in data:
            good_counts[good_to_eat] += 1
            color_counts[good_to_eat][color] += 1
            softness_counts[good_to_eat][softness] += 1

        total = len(data)

        for good_to_eat in GoodToEat:
            self.good_to_eat_prior[good_to_eat] = (
                good_counts[good_to_eat] / total
            )

        for good_to_eat in GoodToEat:
            for color in Color:
                self.color_given_good_to_eat_pmf[good_to_eat][color] = (
                    color_counts[good_to_eat][color] / good_counts[good_to_eat]
                )

        for good_to_eat in GoodToEat:
            for softness in Softness:
                self.softness_given_good_to_eat_pmf[good_to_eat][softness] = (
                    softness_counts[good_to_eat][softness] / good_counts[good_to_eat]
                )

        return self

    """
    Problem 2: use the fields set when the 'fit' method was called to estimate some GoodToEat values
    """
    def predict_color_proba(self,
                            X: List[Color]
                            ) -> List[List[Tuple[GoodToEat, float]]]:

        # imagine predicting a single avocado. What I want you to produce is a pmf over GoodToEat
        # this pmf will be a list that looks like [(GoodToEat.YES, a), (GoodToEat.NO, b)]
        # where 'a' and 'b' are the conditional probabilities Pr[GoodToEat=Yes | Color=c] and Pr[GoodToEat=NO | Color=c]

        # if you have 'n' avocados to predict then I want one pmf *per* avocado. This is why this is a list of lists
        probs_per_example: List[List[Tuple[GoodToEat, float]]] = list()


        for color in X:
            probs = []

            for good_to_eat in GoodToEat:
                prob = (
                    self.color_given_good_to_eat_pmf[good_to_eat][color]
                    * self.good_to_eat_prior[good_to_eat]
                )
                probs.append((good_to_eat, prob))

            total_prob = sum(prob for _, prob in probs)

            normalized_probs = [
                (good_to_eat, prob / total_prob)
                for good_to_eat, prob in probs
            ]

            probs_per_example.append(normalized_probs)

        return probs_per_example

    """
    Problem 2: use the fields set when the 'fit' method was called to estimate some GoodtoEat values
    """
    def predict_softness_proba(self,
                               X: List[Softness]
                               ) -> List[List[Tuple[GoodToEat, float]]]:

        # imagine predicting a single avocado. What I want you to produce is a pmf over GoodToEat
        # this pmf will be a list that looks like [(GoodToEat.YES, a), (GoodToEat.NO, b)]
        # where 'a' and 'b' are the conditional probabilities Pr[GoodToEat=Yes | Softness=s] and Pr[GoodToEat=NO | Softness=s]

        # if you have 'n' avocados to predict then I want one pmf *per* avocado. This is why this is a list of lists
               probs_per_example: List[List[Tuple[GoodToEat, float]]] = list()
    def predict_softness_proba(self,
                               X: List[Softness]
                               ) -> List[List[Tuple[GoodToEat, float]]]:

        probs_per_example: List[List[Tuple[GoodToEat, float]]] = list()

        for softness in X:
            probs = []

            for good_to_eat in GoodToEat:
                prob = (
                    self.softness_given_good_to_eat_pmf[good_to_eat][softness]
                    * self.good_to_eat_prior[good_to_eat]
                )
                probs.append((good_to_eat, prob))

            total_prob = sum(prob for _, prob in probs)

            normalized_probs = [
                (good_to_eat, prob / total_prob)
                for good_to_eat, prob in probs
            ]

            probs_per_example.append(normalized_probs)

        return probs_per_example

    def predict_color(self, X: List[Color]) -> List[GoodToEat]:
        probs_per_example = self.predict_color_proba(X)

        predictions = []

        for probs in probs_per_example:
            prediction = max(probs, key=lambda x: x[1])[0]
            predictions.append(prediction)

        return predictions

    def predict_softness(self, X: List[Softness]) -> List[GoodToEat]:
        probs_per_example = self.predict_softness_proba(X)

        predictions = []

        for probs in probs_per_example:
            prediction = max(probs, key=lambda x: x[1])[0]
            predictions.append(prediction)

        return predictions

In [69]:
m = AvocadoPredictor().fit(data)
pprint(m.good_to_eat_prior)
assert(np.isclose(m.good_to_eat_prior[GoodToEat.YES], 0.52941))  # 9/17 of the training samples are
assert(np.isclose(m.good_to_eat_prior[GoodToEat.NO],  0.47059))  # 8/17 training samples are not good to eat

# TODO: add more of these!
pprint(m.color_given_good_to_eat_pmf)
assert(np.isclose(m.color_given_good_to_eat_pmf[GoodToEat.YES][Color.BLACK],  0.111111))
assert(np.isclose(m.color_given_good_to_eat_pmf[GoodToEat.NO][Color.BLACK],   0.375))


# TODO: add more of these!
pprint(m.softness_given_good_to_eat_pmf)
assert(np.isclose(m.softness_given_good_to_eat_pmf[GoodToEat.YES][Softness.MUSHY],  0.111111))
assert(np.isclose(m.softness_given_good_to_eat_pmf[GoodToEat.NO][Softness.MUSHY],   0.375))


defaultdict(<class 'float'>,
            {<GoodToEat.NO: 1>: 0.47058823529411764,
             <GoodToEat.YES: 0>: 0.5294117647058824})
defaultdict(<function AvocadoPredictor.__init__.<locals>.<lambda> at 0x7a4751264f40>,
            {<GoodToEat.NO: 1>: defaultdict(<class 'float'>,
                                            {<Color.BROWN: 1>: 0.25,
                                             <Color.GREEN: 2>: 0.375,
                                             <Color.BLACK: 0>: 0.375}),
             <GoodToEat.YES: 0>: defaultdict(<class 'float'>,
                                             {<Color.BROWN: 1>: 0.5555555555555556,
                                              <Color.GREEN: 2>: 0.3333333333333333,
                                              <Color.BLACK: 0>: 0.1111111111111111})})
defaultdict(<function AvocadoPredictor.__init__.<locals>.<lambda> at 0x7a47510504a0>,
            {<GoodToEat.NO: 1>: defaultdict(<class 'float'>,
                                          

### Problem 2: Predicting GoodToEat Given a Feature

Now complete the `predict_color_proba` and `predict_softness_proba` methods. These methods are symmetric (i.e. they behave the same way) except `predict_color_proba` processes the `Color` feature while `predict_softness_proba` processes the `Softness` feature (so don't forget to make that change when implementing the two)! Therefore I am only going to describe what I want from the `predict_color_proba` method. Given a `list` of `Color` features (one `Color` feature per avocado), I want you to return an entire pmf over the `GoodToEat` attribute (one pmf per avocado). If the input list has 10 `Color` values then your method should produce 10 pmfs. Each pmf will look like this:

$$[(GoodToEat.YES, Pr[GoodToEat=YES|Color=c]), (GoodToEat.NO, Pr[GoodToEat=No|Color=c])]$$

The way you should calculate the probability $Pr[GoodToEat=x | Color=c]$ is by using Bayes' rule:

$$Pr[GoodToEat=x | Color=c] = \dfrac{Pr[GoodToEat=x]Pr[Color=c|GoodToEat=x]}{Pr[Color=c]}$$

You will need to use the law of total probability to calculate $Pr[Color=c]$ as follows:

$$Pr[Color=c] = \sum\limits_{x\in GoodToEat} Pr[Color=c | GoodToEat=x]Pr[GoodToEat=x]$$

In [70]:
# YOUR CHANGES HERE
m = AvocadoPredictor().fit(data)

# TODO: add more of these
black_prediction = m.predict_color_proba([Color.BLACK])
pprint(black_prediction)
assert(black_prediction[0][0][0] == GoodToEat.YES)
assert(black_prediction[0][1][0] == GoodToEat.NO)


# TODO: add more of these
soft_prediction = m.predict_softness_proba([Softness.SOFT])
pprint(soft_prediction)
assert(soft_prediction[0][0][0] == GoodToEat.YES)
assert(soft_prediction[0][1][0] == GoodToEat.NO)

[[(<GoodToEat.YES: 0>, 0.25), (<GoodToEat.NO: 1>, 0.7499999999999999)]]
[[(<GoodToEat.YES: 0>, 0.8571428571428571),
  (<GoodToEat.NO: 1>, 0.14285714285714285)]]


### Problem 3: Making a Decision

Now complete the `predict_color` and `predict_softness` methods. These method are symmetric (i.e. they behave the same way) except `predict_color` processes the `Color` feature while `predict_softness` processes the `Softness` feature (so don't forget to make that change when implementing the two)! Therefore I am only going to describe what I want from the `predict_color` method. Given a `list` of `Color` features (one `Color` feature per avocado), I want you to return a single `GoodToEat` value. If the input list has 10 `Color` values then your method should produce 10 `GoodToEat` values. The `GoodToEat` value you should produce for a single avocado is the most likely `GoodToEat` value given the provided `Color` value. I would recommend calling `predict_color_proba` and than returning whichever `GoodToEat` value has the largest probability.

Below is a function which will evaluate the accuracy of your model

In [71]:
def accuracy(predictions: List[GoodToEat],
             actual: List[GoodToEat]
             ) -> float:
    if len(predictions) != len(actual):
        raise ValueError(f"ERROR: expected predictions and actual to be same length but got pred={len(predictions)}" +
            " and actual={len(actual)}")

    num_correct: float = 0
    for pred, act in zip(predictions, actual):
        num_correct += int(pred == act)

    return num_correct / len(predictions)

In [72]:
# YOUR CHANGES HERE
m = AvocadoPredictor().fit(data)

color_data: List[Color] = [color for color, _, _ in data]
softness_data: List[Softness] = [softness for _, softness, _ in data]
good_to_eat_data: List[GoodToEat] = [good_to_eat for _, _, good_to_eat in data]

print("Accuracy when predicting only on color: {:.4f}".format(accuracy(m.predict_color(color_data), good_to_eat_data)))
print("Accuracy when predicting only on softness: {:.4f}".format(accuracy(m.predict_softness(softness_data), good_to_eat_data)))

Accuracy when predicting only on color: 0.6471
Accuracy when predicting only on softness: 0.8235
